# Step 10b-E: encode Contriever cho DBPedia-entity, SciFact, NFCorpus (GPU)

- **Accelerator:** GPU T4 ×2. **Internet:** On (tải `facebook/contriever`).
- **Input:** `qudar-beir-data-2`, `qudar-patches` (bản mới).
- **Thời gian ước tính:** SciFact và NFCorpus mỗi bộ 1–2 phút; DBPedia-entity 10 shard × khoảng 9–12 phút (tài liệu ngắn, trung bình 50 từ) → khoảng 1,5–2 giờ. Save Version với khoảng 7 GB output mất thêm 10–20 phút.
- **Sau khi chạy:** tạo dataset `qudar-shards-new` từ thư mục `shards/` (gồm `scifact/`, `nfcorpus/`, `dbpedia-entity/`).
- Bị ngắt giữa chừng: chạy lại ô dưới, các shard đã xong sẽ được bỏ qua.

In [ ]:
import glob, os
DATA2 = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/qudar-beir-data-2/**/dbpedia-entity/corpus.jsonl", recursive=True)[0]))
PATCH = os.path.dirname(glob.glob("/kaggle/input/**/encode_shards.py", recursive=True)[0])
print(DATA2, PATCH)
for ds in ["scifact", "nfcorpus", "dbpedia-entity"]:  # bộ nhỏ trước để kiểm tra nhanh
    !python {PATCH}/encode_shards.py --corpus {DATA2}/{ds}/corpus.jsonl --out /kaggle/working/shards/{ds} --batch-size 256

In [ ]:
# Kiểm tra: số id trong shard phải bằng số tài liệu ở Step 10a (5183, 3633, 4635922)
import json
for ds, n in [("scifact", 5183), ("nfcorpus", 3633), ("dbpedia-entity", 4635922)]:
    files = sorted(glob.glob(f"/kaggle/working/shards/{ds}/shard_*[0-9].ids.json"))
    total = sum(len(json.load(open(f))) for f in files)
    print(f"{ds}: {len(files)} shard, {total} id, khớp Step 10a: {total == n}")
!du -sh /kaggle/working/shards/*